<a href="https://colab.research.google.com/github/zeinab-salah/COSC726-Agentic-Artificial-Intelligence/blob/main/week01_lab0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

import importlib.util
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

print(f"Python {platform.python_version()} on {platform.system()} ({platform.machine()})")
print("PASS · Python 3.11+" if sys.version_info >= (3, 11)
      else "WARN · install Python 3.11 or newer before Lab 1")
print(f"Environment: {'Google Colab (managed)' if IN_COLAB else 'local machine'}")
print(f"Working directory: {Path.cwd()}")

Python 3.12.13 on Linux (x86_64)
PASS · Python 3.11+
Environment: Google Colab (managed)
Working directory: /content


In [2]:
# Isolation check. Colab already gives you a clean managed runtime, so a venv is
# only expected when you are running locally.
if IN_COLAB:
    print("PASS · Colab runtime is isolated and disposable — no virtualenv needed here.")
    print("       Note: Colab RESETS between sessions. GitHub is your permanent record.")
    in_venv = True          # not applicable in Colab; treated as satisfied
else:
    in_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    if in_venv:
        print(f"PASS · virtual environment active: {sys.prefix}")
    else:
        print("WARN · no virtual environment detected")
        if platform.system() == "Windows":
            print(r"      create: python -m venv .venv    activate: .venv\Scripts\activate")
        else:
            print("      create: python3 -m venv .venv    activate: source .venv/bin/activate")

PASS · Colab runtime is isolated and disposable — no virtualenv needed here.
       Note: Colab RESETS between sessions. GitHub is your permanent record.


In [3]:
def run_text(command: list[str]) -> str:
    try:
        result = subprocess.run(command, capture_output=True, text=True, check=False)
        return (result.stdout or result.stderr).strip()
    except OSError as exc:
        return f"ERROR: {exc}"

git_ok = shutil.which("git") is not None
if git_ok:
    print("PASS ·", run_text(["git", "--version"]))
    name = run_text(["git", "config", "user.name"])
    email = run_text(["git", "config", "user.email"])
    if name and email and not name.startswith("ERROR"):
        print(f"PASS · Git identity: {name} <{email}>")
    else:
        print("WARN · set your identity:")
        print('      git config --global user.name "Your Name"')
        print('      git config --global user.email "you@university.ac.uk"')
else:
    print("WARN · Git is not on PATH")

print("PASS · kernel executable:", sys.executable)

PASS · git version 2.34.1
WARN · set your identity:
      git config --global user.name "Your Name"
      git config --global user.email "you@university.ac.uk"
PASS · kernel executable: /usr/bin/python3


In [4]:
MOCK_TRACE = [
    (1, "USER", "Hi, I'm Layla. Where is my order #A1032? It was meant to arrive Tuesday."),
    (2, "DECIDE", "I need the current order status. Use the read-only lookup_order tool."),
    (3, "ACT", 'lookup_order(order_id="A1032")'),
    (4, "OBSERVE", '{"order_id":"A1032","item":"desk lamp","status":"delayed at depot",'
                   '"promised_day":"Tuesday","new_eta":"Friday"}'),
    (5, "DECIDE", "The delay is three calendar days. Check the policy before mentioning compensation."),
    (6, "ACT", 'get_late_delivery_policy(topic="goodwill credit")'),
    (7, "OBSERVE", '{"eligibility":"delay of 3 or more calendar days",'
                   '"remedy":"10% goodwill credit","approval_required":true}'),
    (8, "DECIDE", "The order appears eligible. Applying a credit changes the account, so request "
                  "human approval rather than claim it happened."),
    (9, "ACT", 'request_approval(action="apply_goodwill_credit", order_id="A1032", amount_percent=10)'),
    (10, "OBSERVE", '{"approval_id":"APR-2048","status":"pending","account_changed":false}'),
    (11, "DECIDE", "I have verified the facts and opened the required approval request. Stop and "
                   "report the status, evidence, and next step."),
    (12, "RESPOND", "Hi Layla - order #A1032 is delayed at the depot and is now due Friday. Because "
                    "that is three calendar days late, it appears eligible for a 10% goodwill credit. "
                    "I submitted approval request APR-2048; no account change has been made yet."),
]

def play_trace(upto: int | None = None) -> None:
    events = MOCK_TRACE if upto is None else MOCK_TRACE[:upto]
    width = max(len(kind) for _, kind, _ in MOCK_TRACE)
    for step, kind, text in events:
        print(f"[{kind:<{width}}] step {step:>2}: {text}")
        print("-" * 108)

play_trace()

[USER   ] step  1: Hi, I'm Layla. Where is my order #A1032? It was meant to arrive Tuesday.
------------------------------------------------------------------------------------------------------------
[DECIDE ] step  2: I need the current order status. Use the read-only lookup_order tool.
------------------------------------------------------------------------------------------------------------
[ACT    ] step  3: lookup_order(order_id="A1032")
------------------------------------------------------------------------------------------------------------
[OBSERVE] step  4: {"order_id":"A1032","item":"desk lamp","status":"delayed at depot","promised_day":"Tuesday","new_eta":"Friday"}
------------------------------------------------------------------------------------------------------------
[DECIDE ] step  5: The delay is three calendar days. Check the policy before mentioning compensation.
----------------------------------------------------------------------------------------------------

In [5]:
my_annotations = {
    1: "sense", 2: "reason", 3: "act", 4: "observe", 5: "reason", 6: "act",
    7: "observe", 8: "reason", 9: "act", 10: "observe", 11: "reason", 12: "act",
}

ANSWER_KEY = {1: ["sense"], 2: ["reason"], 3: ["act"], 4: ["observe", "sense"],
              5: ["reason"], 6: ["act"], 7: ["observe", "sense"], 8: ["reason"],
              9: ["act"], 10: ["observe", "sense"], 11: ["reason"], 12: ["act"]}

def check_annotations(answers: dict[int, str]) -> int:
    correct = 0
    for step, expected_list in ANSWER_KEY.items():
        expected = set(expected_list)
        got = str(answers.get(step, "")).strip().lower()
        ok = got in expected
        correct += int(ok)
        verdict = "OK" if ok else ("UNANSWERED" if got in {"", "..."}
                                   else "EXPECTED " + " or ".join(sorted(expected)))
        print(f"step {step:>2}: {got or '(blank)':<10} {verdict}")
    print(f"\nScore: {correct}/{len(ANSWER_KEY)}")
    return correct

check_annotations(my_annotations)

step  1: sense      OK
step  2: reason     OK
step  3: act        OK
step  4: observe    OK
step  5: reason     OK
step  6: act        OK
step  7: observe    OK
step  8: reason     OK
step  9: act        OK
step 10: observe    OK
step 11: reason     OK
step 12: act        OK

Score: 12/12


12

In [6]:
TRACE_AUDIT = """
Name: <Zeinab Salah>
Student ID: <521-125833447>
--------------------------------------------------------------------------
1. EVIDENCE. Which observations support the final response? Cite step numbers.
Step 4 confirms the order is delayed and gives the new ETA (Friday), which the
response repeats. Step 7 confirms the 3-day threshold and the 10% goodwill
credit remedy, which the response cites as the reason the order "appears
eligible." Step 10 confirms the approval request was opened (APR-2048) and
that account_changed is false, which the response states directly.

2. PERMISSIONS. Which tools are read-only, and which action could lead to an
   account change? Why is the approval gate appropriate?
lookup_order (step 3) and get_late_delivery_policy (step 6) are read-only
queries; they only retrieve information and cannot modify anything.
apply_goodwill_credit (the action named inside request_approval, step 9) is
the one that could change the customer's account, since it issues a credit.
The approval gate is appropriate because a financial account change is
consequential and hard to reverse cleanly, so a human should confirm it
before it happens rather than the agent applying it unilaterally.

3. STATE CHANGE. What evidence shows the credit has NOT yet been applied?
Step 10\'s observation explicitly reports "account_changed": false, and the
final response in step 12 states plainly that "no account change has been
made yet." Both are direct, machine-checkable confirmations, not inference.

4. TERMINATION. Why is step 11 a valid stop decision? Name one condition that
   should instead cause escalation or another loop iteration.
Step 11 is valid because every fact needed to answer Layla has been gathered
(order status, policy eligibility, approval status) and the one action that
requires a human has already been correctly deferred - there is nothing left
for the agent to do on its own authority. It should instead escalate or loop
again if, for example, request_approval had returned an error or timeout: the
agent would then need to retry or flag the failure rather than silently stop.

5. TRACE LIMITS. Why should DECIDE be treated as a designed rationale summary,
   not as guaranteed access to private model reasoning?
A real model's internal computation is not directly observable or guaranteed
to match any text it produces about its own reasoning. The DECIDE entries here
were authored by the course team to teach the control-flow pattern clearly;
treating them as a literal transcript of a model\'s "thoughts" would overstate
what any trace - real or mock - can actually prove about internal reasoning.
"""

remaining_audit = TRACE_AUDIT.count("TODO")
print(TRACE_AUDIT)
print("PASS · trace audit complete" if remaining_audit == 0
      else f"WARN · {remaining_audit} TODO item(s) remain")


Name: <Zeinab Salah>
Student ID: <521-125833447>
--------------------------------------------------------------------------
1. EVIDENCE. Which observations support the final response? Cite step numbers.
Step 4 confirms the order is delayed and gives the new ETA (Friday), which the
response repeats. Step 7 confirms the 3-day threshold and the 10% goodwill
credit remedy, which the response cites as the reason the order "appears
eligible." Step 10 confirms the approval request was opened (APR-2048) and
that account_changed is false, which the response states directly.

2. PERMISSIONS. Which tools are read-only, and which action could lead to an
   account change? Why is the approval gate appropriate?
lookup_order (step 3) and get_late_delivery_policy (step 6) are read-only
queries; they only retrieve information and cannot modify anything.
apply_goodwill_credit (the action named inside request_approval, step 9) is
the one that could change the customer's account, since it issues a credit

In [7]:
DESIGN_WORKSHEET = """
PEAS
----
Performance measure: Accurate order status relayed to the customer; correct
                      eligibility determination against the delay policy; zero
                      unauthorized account changes; fast, clear communication
                      of status and next steps.
Environment:         An e-commerce order-fulfillment system: order/shipping
                      database, delivery-policy rules, a human-approval queue,
                      and a customer chat channel.
Actuators:           Sending chat replies to the customer; calling
                      lookup_order; calling get_late_delivery_policy; calling
                      request_approval.
Sensors:              Incoming customer messages; tool-call results returned
                      into the transcript (order data, policy data, approval
                      status).

AUTONOMY AND RISK
-----------------
Starting autonomy level (1-4):                     Level 3
Evidence from the trace (cite step numbers):       Steps 3 and 6 show the
  agent calling read-only tools on its own initiative, with no human asked
  first. Steps 8-9 show the boundary of that autonomy: the agent decides the
  customer is eligible for a credit but does not execute the change itself -
  it requests human approval instead, which is the defining trait of Level 3
  (autonomous action, human sign-off for consequential/irreversible steps).
Highest-risk action in the trace:                  The credit application
  named inside request_approval at step 9 (apply_goodwill_credit) - it is the
  only action in the trace with a real financial/account impact.
What evidence would justify moving up one level?   A track record showing
  approval requests of this type are consistently granted, a defined dollar
  threshold below which auto-approval is low-risk, a reliable way to reverse
  an incorrect credit, and audit logging so every auto-applied credit remains
  reviewable after the fact.
"""

remaining_peas = DESIGN_WORKSHEET.count("TODO")
print(DESIGN_WORKSHEET)
print("PASS · design worksheet complete" if remaining_peas == 0
      else f"WARN · {remaining_peas} TODO item(s) remain")


PEAS
----
Performance measure: Accurate order status relayed to the customer; correct
                      eligibility determination against the delay policy; zero
                      unauthorized account changes; fast, clear communication
                      of status and next steps.
Environment:         An e-commerce order-fulfillment system: order/shipping
                      database, delivery-policy rules, a human-approval queue,
                      and a customer chat channel.
Actuators:           Sending chat replies to the customer; calling
                      lookup_order; calling get_late_delivery_policy; calling
                      request_approval.
Sensors:              Incoming customer messages; tool-call results returned
                      into the transcript (order data, policy data, approval
                      status).

AUTONOMY AND RISK
-----------------
Starting autonomy level (1-4):                     Level 3
Evidence from the trace (cite step n

In [8]:
DECISION_MEMO = """
COSC726 · Lab 0 decision memo
Name: <Zeinab Salah>
--------------------------------------------------------------------------
1. BETTER THAN WHAT?  What is the baseline this agent should be compared with
   (e.g. a fixed auto-reply template)? What does the trace do that the baseline
   cannot?
The baseline is a static auto-reply/FAQ bot that returns a generic "delays
happen, contact support" message with no order-specific data. This agent
looks up the actual order, checks it against a real policy threshold, and
starts a concrete approval workflow instead of a dead end - the baseline
cannot personalize the answer or take any next step on the customer's behalf.

2. MEASURED BY WHAT?  Propose ONE measurable criterion for whether this agent
   handled Layla well. State how you would compute it.
Criterion: the fraction of responses whose claims are fully supported by the
tool observations in the same transcript, with no unauthorized state change.
Compute it by diffing each sentence of the final response against the
OBSERVE events preceding it, and separately checking that no ACT event
changed account state without a corresponding human-approval confirmation.

3. AT WHAT COST?  Count the tool calls and events. If each step involved a model
   call over a growing transcript, where would the cost grow fastest?
There are 3 tool calls (steps 3, 6, 9) and 12 events total. If every DECIDE
and RESPOND step were a real model call, cost would grow fastest toward the
end of the conversation, because each later call re-sends the full growing
transcript (all prior USER/OBSERVE/DECIDE text) as context - token cost per
call rises roughly with conversation length, not just call count.

4. UNDER WHAT FAILURE CONDITIONS?  Pick one step and describe what the agent
   should do if that tool returned an error or a timeout instead.
Step 9, request_approval: if it errored or timed out, the agent must not
assume approval was granted (or denied). It should retry with backoff, and if
it still fails, tell Layla honestly that the credit request could not be
submitted yet and that it will follow up - never state that a request was
opened when it was not.

5. HOW REPRODUCIBLE?  Could a marker clone your repo and reproduce today's
   output exactly? Name the two things most likely to break that.
Yes in principle, since the trace is fully deterministic with no external
state. The two most likely breakers are: (a) an unpinned or ignored
requirements.txt causing a different library/Python version to run the code,
and (b) running the script from the wrong working directory so relative
paths or module imports resolve differently than on the author's machine.

6. WHAT ENGINEERING DECISION FOLLOWS?  State one concrete change you would make
   before this agent went anywhere near a real customer.
Add structured logging (timestamps, request/run IDs, tool latency) plus an
automated test that fails the build if any trace ever reports
account_changed=true without a matching human-approval confirmation event -
turning the "restraint" seen in this trace into an enforced guarantee rather
than a hopeful convention.
"""

remaining_memo = DECISION_MEMO.count("TODO")
print(DECISION_MEMO)
print("PASS · decision memo complete" if remaining_memo == 0
      else f"WARN · {remaining_memo} TODO item(s) remain")


COSC726 · Lab 0 decision memo
Name: <Zeinab Salah>
--------------------------------------------------------------------------
1. BETTER THAN WHAT?  What is the baseline this agent should be compared with
   (e.g. a fixed auto-reply template)? What does the trace do that the baseline
   cannot?
The baseline is a static auto-reply/FAQ bot that returns a generic "delays
happen, contact support" message with no order-specific data. This agent
looks up the actual order, checks it against a real policy threshold, and
starts a concrete approval workflow instead of a dead end - the baseline
cannot personalize the answer or take any next step on the customer's behalf.

2. MEASURED BY WHAT?  Propose ONE measurable criterion for whether this agent
   handled Layla well. State how you would compute it.
Criterion: the fraction of responses whose claims are fully supported by the
tool observations in the same transcript, with no unauthorized state change.
Compute it by diffing each sentence of the 

In [9]:
score = check_annotations(my_annotations)

checks = {
    "Python 3.11+":               sys.version_info >= (3, 11),
    "Isolated environment":       in_venv,
    "Git available":              git_ok,
    "Annotations 12/12":          score == 12,
    "Trace audit complete":       TRACE_AUDIT.count("TODO") == 0,
    "PEAS / autonomy complete":   DESIGN_WORKSHEET.count("TODO") == 0,
    "Decision memo complete":     DECISION_MEMO.count("TODO") == 0,
}

print()
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'WARN'} · {label}")

print()
if all(checks.values()):
    print("READY TO SUBMIT — now follow the push steps in the next cell.")
else:
    print("Resolve the warnings above, or document the setup issue with a demonstrator.")

step  1: sense      OK
step  2: reason     OK
step  3: act        OK
step  4: observe    OK
step  5: reason     OK
step  6: act        OK
step  7: observe    OK
step  8: reason     OK
step  9: act        OK
step 10: observe    OK
step 11: reason     OK
step 12: act        OK

Score: 12/12

PASS · Python 3.11+
PASS · Isolated environment
PASS · Git available
PASS · Annotations 12/12
PASS · Trace audit complete
PASS · PEAS / autonomy complete
PASS · Decision memo complete

READY TO SUBMIT — now follow the push steps in the next cell.
